# 02 – Graph Load
**Techtators — Big Data and Business Intelligence**

Loads the full property graph schema into Neo4j via `LOAD CSV + MERGE`.

Pipeline is **idempotent**: `docker compose down -v` → re-run → same counts every time.

_S3 Hands-On 2 — Added by Rauf_

In [ ]:
# ── 0. Connect to Neo4j ───────────────────────────────────────────────────────
from neo4j import GraphDatabase

URI      = "bolt://localhost:7687"
USER     = "neo4j"
PASSWORD = "password"   # change if needed

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print("✅ Connected to Neo4j")

In [ ]:
# ── Helper: run a Cypher query and return a DataFrame ─────────────────────────
import pandas as pd

def run(cypher, params=None):
    with driver.session() as s:
        result = s.run(cypher, params or {})
        return pd.DataFrame([r.data() for r in result])

def run_write(cypher, params=None):
    with driver.session() as s:
        s.run(cypher, params or {})

print("✅ Helper functions ready")

In [ ]:
# ── 1. Create Constraints (idempotent — IF NOT EXISTS) ─────────────────────────
constraints = [
    "CREATE CONSTRAINT movie_id IF NOT EXISTS FOR (m:Movie) REQUIRE m.movie_id IS UNIQUE",
    "CREATE CONSTRAINT director_id IF NOT EXISTS FOR (d:Director) REQUIRE d.director_id IS UNIQUE",
    "CREATE CONSTRAINT actor_id IF NOT EXISTS FOR (a:Actor) REQUIRE a.actor_id IS UNIQUE",
    "CREATE CONSTRAINT genre_id IF NOT EXISTS FOR (g:Genre) REQUIRE g.genre_id IS UNIQUE",
]

for c in constraints:
    run_write(c)

print("✅ Constraints created")

In [ ]:
# ── 2. Load Movie Nodes ───────────────────────────────────────────────────────
# One LOAD CSV + MERGE block per node label, keyed on id

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///Movie_nodes.csv' AS row
    MERGE (m:Movie {movie_id: toInteger(row.movie_id)})
    SET m.title              = row.title,
        m.release_year       = toInteger(row.release_year),
        m.runtime            = toFloat(row.runtime),
        m.budget             = toFloat(row.budget),
        m.revenue            = toFloat(row.revenue),
        m.imdb_rating        = toFloat(row.imdb_rating),
        m.vote_count         = toInteger(row.vote_count),
        m.original_language  = row.original_language,
        m.overview           = row.overview,
        m.tagline            = row.tagline,
        m.release_date       = row.release_date,
        m.roi                = toFloat(row.roi)
""")

print("✅ Movie nodes loaded")

In [ ]:
# ── 3. Load Director Nodes ────────────────────────────────────────────────────
# Nodes must be loaded BEFORE their relationships

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///Director_nodes.csv' AS row
    MERGE (d:Director {director_id: toInteger(row.director_id)})
    SET d.name = row.name
""")

print("✅ Director nodes loaded")

In [ ]:
# ── 4. Load Actor Nodes ───────────────────────────────────────────────────────

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///Actor_nodes.csv' AS row
    MERGE (a:Actor {actor_id: toInteger(row.actor_id)})
    SET a.name = row.name
""")

print("✅ Actor nodes loaded")

In [ ]:
# ── 5. Load Genre Nodes ───────────────────────────────────────────────────────

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///Genre_nodes.csv' AS row
    MERGE (g:Genre {genre_id: toInteger(row.genre_id)})
    SET g.name = row.name
""")

print("✅ Genre nodes loaded")

In [ ]:
# ── 6. Load DIRECTED Relationships (Director -> Movie) ────────────────────────
# Nodes are loaded first — MATCH can now bind them

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///DIRECTED.csv' AS row
    MATCH (d:Director {director_id: toInteger(row.src_id)})
    MATCH (m:Movie    {movie_id:    toInteger(row.dst_id)})
    MERGE (d)-[:DIRECTED]->(m)
""")

print("✅ DIRECTED relationships loaded")

In [ ]:
# ── 7. Load ACTED_IN Relationships (Actor -> Movie) ───────────────────────────

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///ACTED_IN.csv' AS row
    MATCH (a:Actor {actor_id: toInteger(row.src_id)})
    MATCH (m:Movie {movie_id: toInteger(row.dst_id)})
    MERGE (a)-[:ACTED_IN]->(m)
""")

print("✅ ACTED_IN relationships loaded")

In [ ]:
# ── 8. Load BELONGS_TO Relationships (Movie -> Genre) ─────────────────────────

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///BELONGS_TO.csv' AS row
    MATCH (m:Movie {movie_id: toInteger(row.src_id)})
    MATCH (g:Genre {genre_id: toInteger(row.dst_id)})
    MERGE (m)-[:BELONGS_TO]->(g)
""")

print("✅ BELONGS_TO relationships loaded")

In [ ]:
# ── 9. Load COLLABORATED_WITH Relationships (Director -> Actor) ───────────────

run_write("""
    LOAD CSV WITH HEADERS FROM 'file:///COLLABORATED_WITH.csv' AS row
    MATCH (d:Director {director_id: toInteger(row.src_id)})
    MATCH (a:Actor    {actor_id:    toInteger(row.dst_id)})
    MERGE (d)-[r:COLLABORATED_WITH]->(a)
    SET r.movie_count    = toInteger(row.movie_count),
        r.total_revenue  = toFloat(row.total_revenue)
""")

print("✅ COLLABORATED_WITH relationships loaded")

In [ ]:
# ── 10. POST-LOAD COUNT (Idempotency Receipt) ─────────────────────────────────
# Commit these counts — they are the proof that the pipeline ran correctly.
# Re-running must produce the same numbers every time.

print("📊 Node counts:")
node_counts = run("MATCH (n) RETURN labels(n) AS label, count(*) AS n ORDER BY n DESC")
print(node_counts.to_string(index=False))

print("\n📊 Relationship counts:")
rel_counts = run("MATCH ()-[r]->() RETURN type(r) AS rel, count(*) AS n ORDER BY n DESC")
print(rel_counts.to_string(index=False))

In [ ]:
# ── 11. Sanity Check — Non-zero rows ─────────────────────────────────────────
# Quick check that data is actually in the graph

result = run("MATCH (n) RETURN count(n) AS total_nodes")
total = result['total_nodes'][0]

assert total > 0, "❌ Graph is empty! Check CSV file paths."
print(f"✅ Sanity check passed — {total} nodes in graph")

---
## 📊 Three Cypher Queries Tied to the Business Question

**Business Question:** _Which collaboration networks between actors and directors have the greatest impact on the commercial success of films?_

Each query below maps to a specific sub-question.

In [ ]:
# ── Query 1: Which director-actor pairs collaborated the most AND earned the most?
# Sub-question: Do repeated collaborations lead to higher revenue?

q1 = run("""
    MATCH (d:Director)-[r:COLLABORATED_WITH]->(a:Actor)
    WHERE r.movie_count >= 2
    RETURN d.name            AS director,
           a.name            AS actor,
           r.movie_count     AS movies_together,
           r.total_revenue   AS total_revenue
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print("Query 1 — Top Recurring Collaborations by Revenue")
print(q1.to_string(index=False))

In [ ]:
# ── Query 2: Which directors worked with the highest-rated movies?
# Sub-question: Does a director's network quality (avg rating) reflect commercial value?

q2 = run("""
    MATCH (d:Director)-[:DIRECTED]->(m:Movie)
    WITH d.name AS director,
         count(m)                       AS total_movies,
         round(avg(m.imdb_rating), 2)   AS avg_rating,
         round(sum(m.revenue), 0)       AS total_revenue
    WHERE total_movies >= 3
    RETURN director, total_movies, avg_rating, total_revenue
    ORDER BY avg_rating DESC
    LIMIT 10
""")

print("Query 2 — Directors by Average Rating (min 3 films)")
print(q2.to_string(index=False))

In [ ]:
# ── Query 3: Which genre attracts the most actor-director collaborations?
# Sub-question: Do certain genres create stronger collaboration networks?

q3 = run("""
    MATCH (d:Director)-[:DIRECTED]->(m:Movie)-[:BELONGS_TO]->(g:Genre)
    MATCH (a:Actor)-[:ACTED_IN]->(m)
    WITH g.name AS genre,
         count(DISTINCT m)              AS total_movies,
         count(DISTINCT d)              AS unique_directors,
         count(DISTINCT a)              AS unique_actors,
         round(avg(m.revenue), 0)       AS avg_revenue
    RETURN genre, total_movies, unique_directors, unique_actors, avg_revenue
    ORDER BY avg_revenue DESC
    LIMIT 10
""")

print("Query 3 — Genre Collaboration Networks by Revenue")
print(q3.to_string(index=False))

In [ ]:
# ── 12. Inline Graph Visualization ───────────────────────────────────────────
# Shows the blockbuster collaboration network (revenue > $500M)
# Uses yfiles-jupyter-graphs for interactive rendering

try:
    from yfiles_jupyter_graphs import GraphWidget

    with driver.session() as s:
        result = s.run("""
            MATCH (d:Director)-[:COLLABORATED_WITH]->(a:Actor)
            MATCH (d)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a)
            WHERE m.revenue > 500000000
            RETURN d, a, m
            LIMIT 30
        """)
        graph_data = result.graph()

    w = GraphWidget(graph=graph_data)
    w.node_label_mapping = 'title' if 'title' in dir(w) else None
    display(w)
    print("✅ Interactive graph rendered")

except ImportError:
    print("ℹ️  yfiles not installed — falling back to networkx + matplotlib")

    import networkx as nx
    import matplotlib.pyplot as plt

    with driver.session() as s:
        result = s.run("""
            MATCH (d:Director)-[r:COLLABORATED_WITH]->(a:Actor)
            WHERE r.movie_count >= 2
            RETURN d.name AS director, a.name AS actor, r.movie_count AS w
            LIMIT 20
        """)
        edges = [(row['director'], row['actor'], row['w']) for row in result]

    G = nx.DiGraph()
    for d, a, w in edges:
        G.add_edge(d, a, weight=w)

    plt.figure(figsize=(14, 9))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx(G, pos, node_color='steelblue', font_size=8,
                     edge_color='gray', arrows=True, node_size=800)
    plt.title("Director–Actor Collaboration Network (≥2 films together)", fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('collaboration_network.png', dpi=150)
    plt.show()
    print("✅ Static graph saved as collaboration_network.png")

In [ ]:
# ── Close connection ──────────────────────────────────────────────────────────
driver.close()
print("✅ Neo4j connection closed")